In [8]:
# --- ZELLE 1: Imports & Konfiguration ---
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterator
import pandas as pd
import xml.etree.ElementTree as ET

JSONL_PATH = Path(r"default_process\Ablauf2.1 08072026.jsonl")
TOPIC_MODE = "guide"
CASE_KEY_MODE = "workpiece"

def find_jsonl_path(configured_path: Path) -> Path:
    if configured_path.is_absolute():
        candidates = [configured_path]
    else:
        candidates = [Path.cwd() / configured_path]
        candidates.extend(parent / configured_path for parent in Path.cwd().parents)
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"JSONL file not found: {configured_path}")

SOURCE_PATH = find_jsonl_path(JSONL_PATH)
print(f"Input: {SOURCE_PATH}")

Input: C:\Users\olive\Documents\Uni\Forschungsprojekt\default_process\Ablauf2.1 08072026.jsonl


In [9]:
# --- ZELLE 2: Einlesen der JSONL-Datei ---
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            record["_source_line"] = line_number
            # Vorberechnung des Zeitstempels für schnellere Sortierung
            record["_parsed_ts"] = pd.to_datetime(record.get("timestamp"), utc=True, errors="coerce")
            records.append(record)
    return records

records = read_jsonl(SOURCE_PATH)
print(f"Records read: {len(records):,}")

Records read: 10,012


In [10]:
# --- ZELLE 3: Filterung der prozessrelevanten Topics ---
# Connection- und Pairing-Topics sind hier explizit ausgeschlossen
IMPORTANT_TOPIC_PATTERNS = (
    r"ccu/order/(request|response|completed)",
    r"module/v1/ff/(?:NodeRed/)?[^/]+/(order|state)",
    r"fts/v1/ff/[^/]+/(order|state)",
)

MODULE_STATE_EVENT_TYPES = {
    "SVR3QL1089": {"MODULE_DRILL_FINISHED", "MODULE_DROP_FINISHED", "MODULE_PICK_FINISHED"},
    "SVR4H73901": {"MODULE_DROP_FINISHED", "MODULE_PICK_FINISHED"},
    "SVR4H77011": {"MODULE_CHECK_QUALITY_FINISHED", "MODULE_DROP_FINISHED", "MODULE_PICK_FINISHED"},
    "SVR4H88474": {"MODULE_DROP_FINISHED", "MODULE_MILL_FINISHED", "MODULE_PICK_FINISHED"},
    "SVR4H92774": {
        "MODULE_DROP_FINISHED", "MODULE_INPUT_RGB_FINISHED", "MODULE_PICK_FINISHED",
        "MODULE_RGB_NFC_FINISHED", "MODULE_SETSTATUSLED_FINISHED",
    },
}
IMPORTANT_MODULE_ORDER_COMMANDS = {"PICK", "DROP", "MILL", "DRILL", "FIRE", "CHECK_QUALITY", "RGB_NFC"}
IMPORTANT_FTS_COMMANDS = {"DOCK", "PASS", "TURN"}
IMPORTANT_ACTION_STATES = {"FINISHED", "FAILED", "CANCELLED"}

def topic_is_selected(topic: str) -> bool:
    return any(re.fullmatch(pattern, topic) for pattern in IMPORTANT_TOPIC_PATTERNS)

def command_is_important(topic: str, command: Any, state: Any = None) -> bool:
    if not command:
        return False
    command_name = str(command).upper()
    if topic.startswith("module/v1/ff/"):
        if topic.endswith("/state"):
            module_serial = topic.removesuffix("/state").rsplit("/", 1)[-1]
            event_type = f"MODULE_{command_name}_{str(state).upper()}"
            return event_type in MODULE_STATE_EVENT_TYPES.get(module_serial, set())
        return command_name in IMPORTANT_MODULE_ORDER_COMMANDS
    if topic.startswith("fts/v1/ff/"):
        return (
            command_name in IMPORTANT_FTS_COMMANDS
            and str(state).upper() in IMPORTANT_ACTION_STATES
        )
    return True

selected_records = [r for r in records if topic_is_selected(r["topic"])]

In [11]:
# --- ZELLE 4: Extraktionsregeln & Event-Generierung ---
def payload_objects(payload: Any) -> Iterator[dict[str, Any]]:
    if isinstance(payload, dict):
        yield payload
    elif isinstance(payload, list):
        yield from (item for item in payload if isinstance(item, dict))

def find_value(data: Any, key: str) -> str | None:
    if isinstance(data, dict):
        value = data.get(key)
        if value not in (None, "") and not isinstance(value, (dict, list)):
            return str(value)
        for child in data.values():
            found = find_value(child, key)
            if found:
                return found
    elif isinstance(data, list):
        for child in data:
            found = find_value(child, key)
            if found:
                return found
    return None

def workpiece_id(data: Any) -> str | None:
    return find_value(data, "workpieceId") or find_value(data, "loadId")

ORDER_TO_WORKPIECE: dict[str, str] = {}
for record in selected_records:
    if record["topic"] in {"ccu/order/response", "ccu/order/completed"}:
        for payload in payload_objects(record["payload"]):
            order_id = find_value(payload, "orderId")
            piece_id = workpiece_id(payload)
            if order_id and piece_id:
                ORDER_TO_WORKPIECE[order_id] = piece_id

def get_case_id(payload: dict[str, Any]) -> str | None:
    order_id = find_value(payload, "orderId")
    piece_id = workpiece_id(payload)
    if not piece_id and order_id:
        piece_id = ORDER_TO_WORKPIECE.get(order_id)
    if CASE_KEY_MODE == "workpiece" and piece_id:
        return f"workpiece:{piece_id}"
    if order_id:
        return f"order:{order_id}"
    if piece_id:
        return f"workpiece:{piece_id}"
    return None

def event_time(record: dict[str, Any], payload: dict[str, Any], action: dict[str, Any] | None = None):
    candidates = []
    if action:
        candidates.append(action.get("timestamp"))
    candidates.extend([payload.get("timestamp"), payload.get("ts"), record.get("timestamp")])
    for value in candidates:
        if value:
            parsed = pd.to_datetime(value, utc=True, errors="coerce")
            if not pd.isna(parsed):
                return parsed
    return pd.NaT

IGNORED_COMMANDS = {"factsheetRequest", "findInitialDockPosition", "clearLoadHandler"}
events: list[dict[str, Any]] = []

def add_event(
    record: dict[str, Any], payload: dict[str, Any], activity: str,
    lifecycle: str | None = None, action: dict[str, Any] | None = None,
    resource: str | None = None,
) -> None:
    case_id = get_case_id(payload)
    if not case_id:
        return
    timestamp = event_time(record, payload, action)
    if pd.isna(timestamp):
        return
    
    events.append({
        "case_id": case_id,
        "activity": activity,
        "timestamp": timestamp,
        "resource": resource or find_value(payload, "serialNumber"),
        "source_topic": record["topic"], # Nutzt die JSONL-Struktur für das Source Topic
        "lifecycle": lifecycle,
        "order_id": find_value(payload, "orderId"),
        "workpiece_id": workpiece_id(payload),
        "workpiece_type": find_value(payload, "type"),
        "source_line": record["_source_line"],
        "details": json.dumps(action or payload, ensure_ascii=False, separators=(",", ":")),
    })

ordered_records = sorted(selected_records, key=lambda r: r["_parsed_ts"])
fts_driving_state: dict[str, bool] = {}

for record in ordered_records:
    topic = record["topic"]
    for payload in payload_objects(record["payload"]):
        if topic == "ccu/order/request":
            add_event(record, payload, "ORDER_REQUESTED", "REQUESTED", resource="CCU")
        elif topic == "ccu/order/response":
            state = str(payload.get("state", "ACCEPTED")).upper()
            activity = "ORDER_ACCEPTED" if state in {"IN_PROGRESS", "ACCEPTED"} else f"ORDER_RESPONSE_{state}"
            add_event(record, payload, activity, state, resource="CCU")
        elif topic == "ccu/order/completed":
            state = str(payload.get("state", "COMPLETED")).upper()
            activity = "ORDER_CANCELLED" if "CANCEL" in state else "ORDER_COMPLETED"
            add_event(record, payload, activity, state, resource="CCU")
        
        elif topic.startswith("module/v1/ff/"):
            resource = find_value(payload, "serialNumber") or topic.removeprefix("module/v1/ff/").rsplit("/", 1)[0]
            if topic.endswith("/order"):
                action = payload.get("action") or {}
                command = action.get("command") or find_value(action, "type")
                if command and command not in IGNORED_COMMANDS and command_is_important(topic, command):
                    add_event(record, payload, f"MODULE_ORDER_{str(command).upper()}", "ORDERED", action, resource)
            else:
                actions = payload.get("actionStates", payload.get("actionState", []))
                if isinstance(actions, dict): actions = [actions]
                if not isinstance(actions, list): actions = []
                for action in actions:
                    if not isinstance(action, dict): continue
                    command = action.get("command") or find_value(action.get("metadata", {}), "type")
                    state = action.get("state")
                    if command and state and command not in IGNORED_COMMANDS and command_is_important(topic, command, state):
                        add_event(record, payload, f"MODULE_{str(command).upper()}_{str(state).upper()}", str(state), action, resource)
        
        elif topic.startswith("fts/v1/ff/"):
            resource = find_value(payload, "serialNumber") or topic.split("/")[-2]
            if topic.endswith("/order"):
                add_event(record, payload, "FTS_ORDER_SENT", "PLANNED", resource=resource)
            else:
                driving = payload.get("driving")
                if isinstance(driving, bool):
                    previous_driving = fts_driving_state.get(resource)
                    if previous_driving is not None and driving != previous_driving:
                        activity = "FTS_DRIVING_START" if driving else "FTS_DRIVING_STOP"
                        lifecycle = "STARTED" if driving else "STOPPED"
                        add_event(record, payload, activity, lifecycle, resource=resource)
                    fts_driving_state[resource] = driving
                actions = payload.get("actionStates", payload.get("actionState", []))
                if isinstance(actions, dict): actions = [actions]
                if not isinstance(actions, list): actions = []
                for action in actions:
                    if not isinstance(action, dict): continue
                    command, state = action.get("command"), action.get("state")
                    if command and state and command not in IGNORED_COMMANDS and command_is_important(topic, command, state):
                        add_event(record, payload, f"FTS_{str(command).upper()}_{str(state).upper()}", str(state), action, resource)

EVENT_COLUMNS = [
    "case_id", "activity", "timestamp", "resource", "source_topic", "lifecycle",
    "order_id", "workpiece_id", "workpiece_type", "source_line", "details",
]
event_log = pd.DataFrame(events, columns=EVENT_COLUMNS)
event_log["timestamp"] = pd.to_datetime(event_log["timestamp"], utc=True, errors="coerce")
event_log = event_log.sort_values(["timestamp", "source_line"], kind="stable").reset_index(drop=True)

## Duplicate-event review and correction

Duplicate candidates share `activity`, `timestamp`, and `resource`. The diagnostic groups candidate groups by the attributes that differ and reports how often each pattern occurs, with example source lines/topics.

- **Only source provenance differs:** differing `source_line` and/or `source_topic` values are combined into lists. For example, duplicate publications on `module/v1/ff/SVR4H92774/state` and `module/v1/ff/NodeRed/SVR4H92774/state` become one event with both lines and topics retained.
- **Order ID differs together with source provenance:** this is one duplicate pattern, not an order-ID-only pattern. Variations in `order_id`, `source_topic`, and `source_line` can occur together; merge them into lists on one event. Any other differing event attributes are also retained as lists rather than preventing the merge.
- **Other differing attributes:** candidate events can also differ in `case_id`, `workpiece_id`, `details`, lifecycle, or other event attributes. Merge the group and retain each distinct value in a list. Since XES requires one trace key per event, keep the first `case_id` as the representative `case_id` and store every associated case ID in `case_ids`.
- **Identical event rows:** when all attributes match, combine the repeated rows too; `source_line` and `source_topic` still use list fields to keep the provenance schema consistent.

Every output event stores `source_line` and `source_topic` as lists, including single-occurrence events. `order_id` and any other differing attributes are represented as lists when a group contains multiple distinct values. Every group sharing the duplicate key is merged after its difference pattern is reported.

In [12]:
# --- Duplicate-event review and consolidation ---
# Events with the same identity key are treated as duplicate representations of one occurrence.
DUPLICATE_KEY_COLUMNS = ["activity", "timestamp", "resource"]
SOURCE_LINE_COLUMN = "source_line"
SOURCE_TOPIC_COLUMN = "source_topic"
CASE_ID_COLUMN = "case_id"


def unique_values(values: Iterator[Any] | list[Any]) -> list[Any]:
    """Return distinct values in first-seen order, treating missing values as one value."""
    result: list[Any] = []
    for value in values:
        if value is None or (
            not isinstance(value, (list, tuple, dict)) and pd.isna(value)
        ):
            normalized = None
        else:
            normalized = value
        if normalized not in result:
            result.append(normalized)
    return result


# case_ids preserves all trace associations while case_id remains a scalar XES trace key.
duplicate_report: dict[tuple[str, ...], dict[str, Any]] = {}
corrected_rows: list[dict[str, Any]] = []

# dropna=False ensures events with an empty resource are still reviewed together.
for _, duplicate_group in event_log.groupby(
    DUPLICATE_KEY_COLUMNS, sort=False, dropna=False
):
    group = duplicate_group.copy()

    # Report every differing field together. For example, order_id, source_topic,
    # and source_line can all differ in the same duplicate group; this is one signature.
    if len(group) > 1:
        differing_fields = [
            column
            for column in event_log.columns
            if group[column].nunique(dropna=False) > 1
        ]
        signature = tuple(sorted(differing_fields)) or ("no differing attributes",)
        stats = duplicate_report.setdefault(
            signature, {"groups": 0, "rows": 0, "examples": []}
        )
        stats["groups"] += 1
        stats["rows"] += len(group)
        if len(stats["examples"]) < 3:
            stats["examples"].append({
                "source_lines": unique_values(group[SOURCE_LINE_COLUMN].tolist()),
                "source_topics": unique_values(group[SOURCE_TOPIC_COLUMN].tolist()),
            })

    # Merge every candidate group, including byte-for-byte duplicate event rows.
    # Start with one row to retain stable event order and scalar identity fields.
    row = group.iloc[0].to_dict()
    row["case_ids"] = unique_values(group[CASE_ID_COLUMN].tolist())
    for column in event_log.columns:
        values = unique_values(group[column].tolist())
        if column in {SOURCE_LINE_COLUMN, SOURCE_TOPIC_COLUMN}:
            # Always use lists for provenance, including groups with just one source.
            row[column] = values
        elif len(values) > 1 and column != CASE_ID_COLUMN:
            # Preserve all differing order IDs and any other varying event attributes.
            row[column] = values
    corrected_rows.append(row)

# Replace the raw event table before XES export. case_ids is extra provenance; case_id
# remains the representative trace assignment so the exporter can group events safely.
REVIEWED_EVENT_COLUMNS = [*EVENT_COLUMNS, "case_ids"]
event_log = pd.DataFrame(corrected_rows, columns=REVIEWED_EVENT_COLUMNS)
event_log = event_log.sort_values(["timestamp"], kind="stable").reset_index(drop=True)

print("Duplicate candidate groups by differing attribute(s):")
if not duplicate_report:
    print("  None found for matching activity, timestamp, and resource.")
for differing_attributes, stats in sorted(duplicate_report.items()):
    label = ", ".join(differing_attributes)
    print(f"  differing {label}: {stats['groups']} group(s), {stats['rows']} row(s)")
    for example in stats["examples"]:
        print(
            f"    example source_line(s)={example['source_lines']}; "
            f"source_topic(s)={example['source_topics']}"
        )
merged_group_count = sum(stats["groups"] for stats in duplicate_report.values())
print(f"Merged duplicate candidate groups: {merged_group_count}")

Duplicate candidate groups by differing attribute(s):
  differing case_id, source_line, workpiece_id: 48 group(s), 350 row(s)
    example source_line(s)=[827, 828, 833, 840, 843, 844, 852, 853, 858, 879, 892, 893, 947]; source_topic(s)=['fts/v1/ff/1mO4/state']
    example source_line(s)=[852, 853, 858, 879, 892, 893, 947]; source_topic(s)=['fts/v1/ff/1mO4/state']
    example source_line(s)=[892, 893, 947]; source_topic(s)=['fts/v1/ff/1mO4/state']
  differing source_line: 109 group(s), 790 row(s)
    example source_line(s)=[365, 1155, 1973, 2487, 2961, 3739, 3824, 4476, 4748, 5449, 5560, 6334, 6607, 7076, 8321, 9089, 9593, 9877]; source_topic(s)=['ccu/order/completed']
    example source_line(s)=[151, 152, 218]; source_topic(s)=['fts/v1/ff/1mO4/state']
    example source_line(s)=[2961, 3739, 3824, 4476, 4748, 5449, 5560, 6334, 6607, 7076, 8321, 9089, 9593, 9877]; source_topic(s)=['ccu/order/completed']
  differing source_line, source_topic: 31 group(s), 62 row(s)
    example source_line

In [13]:
# --- ZELLE 5: XES-Export (Kein CSV) ---
XES_LOG_PATH = SOURCE_PATH.with_name(f"{SOURCE_PATH.stem}_event_log.xes")

XES_NAMESPACE = "http://www.xes-standard.org/"
ET.register_namespace("", XES_NAMESPACE)
qname = lambda name: f"{{{XES_NAMESPACE}}}{name}"
xes_log = ET.Element(qname("log"), {"xes.version": "1.0", "xes.features": "nested-attributes"})

for prefix, name, uri in [
    ("concept", "Concept", "http://www.xes-standard.org/concept.xesext"),
    ("time", "Time", "http://www.xes-standard.org/time.xesext"),
    ("lifecycle", "Lifecycle", "http://www.xes-standard.org/lifecycle.xesext"),
    ("org", "Organizational", "http://www.xes-standard.org/org.xesext"),
]:
    ET.SubElement(xes_log, qname("extension"), {"name": name, "prefix": prefix, "uri": uri})

ET.SubElement(xes_log, qname("string"), {"key": "concept:name", "value": SOURCE_PATH.stem})
ET.SubElement(xes_log, qname("classifier"), {"name": "Activity", "keys": "concept:name"})

xes_keys = {
    "activity": "concept:name",
    "timestamp": "time:timestamp",
    "resource": "org:resource",
}

for case_id, case_events in event_log.groupby("case_id", sort=True):
    trace = ET.SubElement(xes_log, qname("trace"))
    ET.SubElement(trace, qname("string"), {"key": "concept:name", "value": str(case_id)})
    for _, event in case_events.iterrows():
        xes_event = ET.SubElement(trace, qname("event"))
        for column, value in event.items():
            if column == "case_id" or value is None or (
                not isinstance(value, (list, tuple)) and pd.isna(value)
            ):
                continue
            key = xes_keys.get(column, column)
            
            if isinstance(value, (list, tuple)):
                # Lists preserve every source line/topic/order ID in the exported event.
                tag, text = "string", json.dumps(value, ensure_ascii=False, separators=(",", ":"))
            elif column == "timestamp":
                tag, text = "date", value.isoformat().replace("+00:00", "Z")
            elif column == "source_line":
                tag, text = "int", str(int(value))
            elif column == "source_topic":
                # Das Topic wird in der XES-Datei für jedes Event ausgewiesen
                tag, text = "string", str(value)
            else:
                tag, text = "string", str(value)
                
            ET.SubElement(xes_event, qname(tag), {"key": key, "value": text})

ET.indent(xes_log, space="  ")
ET.ElementTree(xes_log).write(XES_LOG_PATH, encoding="utf-8", xml_declaration=True)
print(f"XES exported: {XES_LOG_PATH}")

XES exported: C:\Users\olive\Documents\Uni\Forschungsprojekt\default_process\Ablauf2.1 08072026_event_log.xes


In [14]:
# --- Event-log metrics for process and time analysis ---
# Summarize the reviewed event_log created by the duplicate-consolidation cell.
# These durations are observed timestamp spans, not necessarily complete process times
# if the captured log begins or ends partway through a case.

metrics_log = event_log.copy()
metrics_log["timestamp"] = pd.to_datetime(metrics_log["timestamp"], utc=True, errors="coerce")

total_events = len(metrics_log)
case_count = metrics_log["case_id"].nunique(dropna=True)
activity_count = metrics_log["activity"].nunique(dropna=True)
resource_count = metrics_log["resource"].nunique(dropna=True)
timestamped_events = int(metrics_log["timestamp"].notna().sum())

print("Event-log overview")
print(f"  Events: {total_events:,}")
print(f"  Cases/traces: {case_count:,}")
print(f"  Event types/activities: {activity_count:,}")
print(f"  Resources: {resource_count:,}")
print(
    f"  Events with timestamps: {timestamped_events:,} "
    f"({timestamped_events / total_events:.1%})"
    if total_events else "  Events with timestamps: 0"
)
if timestamped_events:
    observed_start = metrics_log["timestamp"].min()
    observed_end = metrics_log["timestamp"].max()
    print(f"  Observed log span: {observed_start} to {observed_end} ({observed_end - observed_start})")

# Trace length helps compare process variants and assess trace coverage.
case_sizes = metrics_log.groupby("case_id", dropna=True).size()
if not case_sizes.empty:
    print("\nEvents per case")
    print(
        f"  Mean: {case_sizes.mean():.2f}; median: {case_sizes.median():.2f}; "
        f"min: {case_sizes.min()}; max: {case_sizes.max()}"
    )
    print(f"  90th percentile: {case_sizes.quantile(0.90):.2f}")

# Case duration and waiting time are calculated only from timestamped events within a case.
valid_timestamps = metrics_log.dropna(subset=["case_id", "timestamp"]).copy()
case_durations: list[pd.Timedelta] = []
inter_event_gaps: list[pd.Timedelta] = []
for _, case_events in valid_timestamps.groupby("case_id", sort=False):
    ordered_times = case_events["timestamp"].sort_values(kind="stable")
    if len(ordered_times) >= 2:
        case_durations.append(ordered_times.iloc[-1] - ordered_times.iloc[0])
        inter_event_gaps.extend(ordered_times.diff().dropna().tolist())


def print_duration_summary(label: str, durations: list[pd.Timedelta]) -> None:
    """Print distribution summaries for observed process-time measurements."""
    if not durations:
        print(f"  {label}: not available (no case has at least two timestamped events)")
        return
    seconds = pd.Series([duration.total_seconds() for duration in durations])
    print(
        f"  {label}: n={len(seconds):,}; mean={pd.to_timedelta(seconds.mean(), unit='s')}; "
        f"median={pd.to_timedelta(seconds.median(), unit='s')}; "
        f"p90={pd.to_timedelta(seconds.quantile(0.90), unit='s')}; "
        f"min={pd.to_timedelta(seconds.min(), unit='s')}; "
        f"max={pd.to_timedelta(seconds.max(), unit='s')}"
    )

print("\nTime-analysis metrics (observed timestamps)")
print_duration_summary("Observed case duration (last event - first event)", case_durations)
print_duration_summary("Inter-event gap within cases", inter_event_gaps)
if total_events:
    print(f"  Events missing timestamp: {total_events - timestamped_events:,}")
    print(f"  Events missing case ID: {int(metrics_log['case_id'].isna().sum()):,}")
    print(f"  Events missing resource: {int(metrics_log['resource'].isna().sum()):,}")
    print(f"  Events missing activity: {int(metrics_log['activity'].isna().sum()):,}")

# activity is the notebook's event-type field. Report observed activity counts per resource.
resource_event_types = (
    metrics_log.groupby(["resource", "activity"], dropna=False)
    .size()
    .rename("event_count")
    .reset_index()
    .sort_values(["resource", "activity"], kind="stable", na_position="last")
)
print("\nRecorded event types by resource")
if resource_event_types.empty:
    print("  No events available.")
else:
    for resource, resource_rows in resource_event_types.groupby("resource", dropna=False, sort=True):
        resource_label = "<missing resource>" if pd.isna(resource) else str(resource)
        print(f"\n  {resource_label}:")
        for _, event_type in resource_rows.iterrows():
            activity = "<missing activity>" if pd.isna(event_type["activity"]) else str(event_type["activity"])
            print(f"    {activity}: {int(event_type['event_count']):,}")

# Keep a tabular form for sorting or further downstream analysis.
display(resource_event_types)


Event-log overview
  Events: 484
  Cases/traces: 19
  Event types/activities: 20
  Resources: 8
  Events with timestamps: 484 (100.0%)
  Observed log span: 2026-07-08 08:02:51.637495+00:00 to 2026-07-08 08:33:32.253838+00:00 (0 days 00:30:40.616343)

Events per case
  Mean: 25.47; median: 18.00; min: 1; max: 78
  90th percentile: 56.20

Time-analysis metrics (observed timestamps)
  Observed case duration (last event - first event): n=10; mean=0 days 00:12:38.110803100; median=0 days 00:11:39.910000; p90=0 days 00:17:09.293860300; min=0 days 00:09:02.419000; max=0 days 00:18:54.108061
  Inter-event gap within cases: n=465; mean=0 days 00:00:16.303458131; median=0 days 00:00:06.348057; p90=0 days 00:00:41.145400; min=0 days 00:00:00; max=0 days 00:03:59.401286
  Events missing timestamp: 0
  Events missing case ID: 0
  Events missing resource: 0
  Events missing activity: 0

Recorded event types by resource

  1mO4:
    FTS_DOCK_FINISHED: 62
    FTS_ORDER_SENT: 62
    FTS_PASS_FINISHED: 

,resource,activity,event_count
0,1mO4,FTS_DOCK_FINISHED,62
1,1mO4,FTS_ORDER_SENT,62
2,1mO4,FTS_PASS_FINISHED,34
3,1mO4,FTS_TURN_FINISHED,66
4,CCU,ORDER_ACCEPTED,18
5,CCU,ORDER_COMPLETED,18
6,CCU,ORDER_REQUESTED,9
7,NodeRed/SVR4H92774,MODULE_ORDER_RGB_NFC,9
8,SVR3QL1089,MODULE_DRILL_FINISHED,6
9,SVR3QL1089,MODULE_DROP_FINISHED,6


## Additional filtered, workpiece-aware event log

This section creates a second XES file (`<Ablauf>_filtered_event_log.xes`). It does not replace or modify `event_log` or the original `<Ablauf>_event_log.xes` export above.

- **Trace model:** the complete input Ablauf is one XES trace, named after the input file. Individual `workpiece_id` values remain event attributes; they do not split this filtered log into multiple traces.
- **CCU:** CCU order request, response, and completion events are omitted.
- **Module naming:** serial numbers map to the MQTT guide's module IDs: `SVR3QL1089` = `DRILL`, `SVR4H88474` = `MILL`, `SVR4H77011` = `AIQS`, `SVR4H92774` = `DPS`, and `SVR4H73901` = `HBW`. Activity names are readable descriptions rather than command-shaped identifiers. Order events describe the requested operation (for example, `Order created: Drill the workpiece`, `Order created: Store the workpiece in the HBW`, or `Order created: Inspect the workpiece quality`).
- **Production modules:** selected finished-state events use natural descriptions such as `Drilling started`, `Milling completed`, `Workpiece stored in the HBW`, and `Workpiece retrieved from the HBW`. DPS events describe factory input or finished-product shipment. Other module action-state events are omitted.
- **FTS:** transport is represented by two point events per workpiece, not by one event describing the whole journey. Names state both endpoints (for example, `Transport departed from DPS to HBW` and `Transport arrived at HBW from DPS`); the destination in both names is the final loading/unloading module, never an intermediate node. The start timestamp is when the FTS begins driving with the workpiece loaded. The end timestamp is the arrival stop with the workpiece still loaded and `waitingForLoadHandling` true. Both events share an `FTS_Trip_ID`; `origin`/`destination` hold the endpoints and `route` holds only intermediate node IDs. A stop between route segments does not end the transport. Same-location docking/handling cycles without route progress are not emitted as transports. Loads are unrolled into one event pair per workpiece; a start is retained if the Ablauf ends before arrival is observed.
- Repeated normalized events are consolidated by activity, timestamp, resource, workpiece, and trip, while differing source provenance is retained.


In [ ]:
# --- Additive filtered XES export ---
try:
    from event_log_creation.filtered_event_log import (
        build_filtered_event_log,
        export_filtered_xes,
    )
except ModuleNotFoundError as error:
    if error.name != "event_log_creation":
        raise
    from filtered_event_log import build_filtered_event_log, export_filtered_xes

FILTERED_XES_LOG_PATH = SOURCE_PATH.with_name(
    f"{SOURCE_PATH.stem}_filtered_event_log.xes"
)
filtered_event_log = build_filtered_event_log(
    ordered_records,
    case_id=SOURCE_PATH.stem,
    order_to_workpiece=ORDER_TO_WORKPIECE,
)

if not filtered_event_log.empty:
    assert filtered_event_log["case_id"].nunique(dropna=True) == 1
    assert not filtered_event_log["source_topic"].astype(str).str.startswith("ccu/").any()
    assert not filtered_event_log["activity"].astype(str).str.startswith("ORDER_").any()

export_filtered_xes(
    filtered_event_log,
    FILTERED_XES_LOG_PATH,
    case_id=SOURCE_PATH.stem,
)
print(f"Filtered XES exported: {FILTERED_XES_LOG_PATH}")
print(f"  Events: {len(filtered_event_log):,}")
print(f"  Ablauf traces: 1 (case_id={SOURCE_PATH.stem})")
print(
    "  Workpieces: "
    f"{filtered_event_log['workpiece_id'].nunique(dropna=True):,}"
)
print(
    "  FTS trip IDs: "
    f"{filtered_event_log['FTS_Trip_ID'].nunique(dropna=True):,}"
)
